# Queen Editor → Video (WAN 2.2 I2V) — Colab

`prompt_converter.ipynb`'ın ürettiği **`video.json`**'u alır, Queen Editor'ün proje klasöründeki fotoğrafları o dosyadaki hareket prompt'larıyla videoya çevirir. ComfyUI arka planda **API** olarak çalışır; **UI açılmaz, tünel yok.**

```
video.json  ──yükle──>  [ bu notebook ]  ──>  MyDrive/queen-tools/<proje>/1_a.mp4 …
                              │
                    fotoğrafları Queen Editor'ün
                    klasöründen OKUR (oraya yazmaz)
```

**Fotoğraflar kopyalanmaz.** Yol JSON'un içindeki `folder` alanından gelir; notebook o klasörü yalnız okur, içine hiçbir şey yazmaz.

**Ham export dosyası da verilebilir** — iki dosyanın şekli aynı. O zaman videolar foto prompt'uyla üretilir.

```
MyDrive/queen-tools/
├── workflow_api.json   ← grafın kopyası (bir kez konur)
└── <proje>/
    ├── video.json      ← prompt_converter yazar
    └── 1_a.mp4 …       ← bu notebook yazar
```

> **Graf nereden gelir:** repodaki `collab-toolbox/video_generator/wan22-arbuzai/workflow_api.json`'u indirip Drive'da `queen-tools/` altına koy. Grafı değiştirmek istersen `wan22-arbuzai/manual.ipynb` → **Workflow → Export (API)**.

**Gerekenler:** **A100** runtime · Colab **Secrets**'ta `CIVITAI_COOKIE` (notebook erişimi açık).

Sıra:
1. **CONFIG** — Drive mount + graf/cookie ayarları
2. **Plan dosyasını yükle** — `video.json` · ardından **üretim planı** basılır
3. **Ortak Yardımcılar** — log + fail-loud run + model doğrulama
4. **ComfyUI + custom node'lar** (16)
5. **Modeller** — önce gated probe, sonra indir (~36 GiB)
6. **ComfyUI'yi başlat** (arka planda, API)
7. **Üret** — her kare: yükle, render et, Drive'a yaz

> **Yarıda kalırsa baştan çalıştır.** Çıktısı olan atlanır, kalanlar üretilir. Yeniden üretmek için Drive'dan o mp4'ü sil.

> **Video pahalı.** Her render A100'de dakikalar sürer; toplam = kare sayısı × `VARIANTS`.

> **LoRA'lar grafikte.** `wan22-arbuzai/manual.ipynb` + UI'da ayarlayıp Export (API) ile dondurursun; bu notebook onlara dokunmaz, yalnız fotoğraf/prompt/seed yazar.

## 1) CONFIG

Google Drive **burada** mount edilir: auth istemi ilk saniyede çıksın, 36 GiB'lık model indirmesinin ortasında seni beklemesin.

Burada doldurulacak bir prompt listesi **yok** — prompt'lar bir sonraki bölümde yükleyeceğin `video.json`'dan gelir.

`VARIANTS` her fotoğraf için kaç video üretileceğidir (farklı seed). Video pahalı olduğu için 1.

Civitai cookie'si **Colab Secrets**'tan okunur (`CIVITAI_COOKIE`) — notebook'un içinde durmaz. Süresi ~30 günde dolar; bittiğinde `civitai.red`'den yeni değeri alıp **sırrı** güncelle.

In [ ]:
# === Google Drive — en başta mount edilir ===
# The auth prompt has to appear in the first second, not in the middle of a 36 GiB download.
from google.colab import drive, userdata
drive.mount('/content/drive')

SEED     = None              # None -> a fresh random seed per variant; a number -> SEED + v
VARIANTS = 1                 # videos per photo (different seeds) -- video is expensive

# === Drive ===
DRIVE_ROOT        = "/content/drive/MyDrive/queen-tools"
WORKFLOW_FILENAME = "workflow_api.json"      # under DRIVE_ROOT, API format

# === Civitai login-gated download ===
# Read from Colab Secrets under the same name Queen Editor uses, so one paste serves both tools and
# the token is not committed with the notebook.
# How to get the value: civitai.red -> log in -> F12 -> Application -> Cookies -> __Secure-civ-token
# (double-click -> Ctrl+A -> Ctrl+C; a single click truncates it and the len > 200 gate still passes).
# NOTE: auth moved to auth.civitai.com -> the cookie NAME is __Secure-civ-token (NOT the old
#   __Secure-civitai-token) and the value is a short ES256 JWT (~420 chars), not the old long JWE.
# Cookie only; never a ?token= API key -> gated assets answer 401.
COOKIE_VALUE = userdata.get('CIVITAI_COOKIE')

# === Render ===
TIMEOUT_PER_RENDER = 30 * 60   # seconds -- a video that misses this fails loud
POLL_INTERVAL      = 5         # seconds -- /history poll interval

# === Derived paths ===
COMFY_PORT       = 8188
COMFYUI_URL      = f"http://127.0.0.1:{COMFY_PORT}"
WORKFLOW_PATH    = f"{DRIVE_ROOT}/{WORKFLOW_FILENAME}"

COMFY_ROOT       = "/content/ComfyUI"
COMFY_OUTPUT_DIR = f"{COMFY_ROOT}/output"
COMFY_LOG        = "/content/comfyui.log"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
# Two asserts, not one: a missing secret and a truncated value are different mistakes and a single
# message could not name which one happened.
assert COOKIE_VALUE, "❌ CIVITAI_COOKIE okunamadı — Colab Secrets'a ekle ve 'Notebook access' aç"
assert len(COOKIE_VALUE) > 200, f"❌ CIVITAI_COOKIE çok kısa ({len(COOKIE_VALUE)} karakter) — değer kırpılmış, civitai.red'den __Secure-civ-token'ı çift tıklayıp tamamını kopyala"
assert os.path.exists(WORKFLOW_PATH), f"❌ Workflow yok: {WORKFLOW_PATH} — repodaki wan22-arbuzai/workflow_api.json'u buraya kopyala"
assert VARIANTS >= 1, "❌ VARIANTS en az 1 olmalı"

print(f"✓ Drive: {DRIVE_ROOT}")
print(f"✓ Cookie: Secrets'tan okundu ({len(COOKIE_VALUE)} char)  |  Timeout: {TIMEOUT_PER_RENDER // 60} dk/video")
print(f"✓ Seed: {SEED if SEED is not None else 'varyant başına rastgele'}  |  Varyant: {VARIANTS}")
print("=== GPU ===")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2) Plan dosyasını yükle

`prompt_converter.ipynb`'ın indirdiği `video.json`'u buraya yükle. **İş emri yüklediğin dosyadır** — Drive'dan seçim yapılmaz, böylece bayat bir ayar yanlış projeyi render edemez.

Ham `<proje>-export.json` de yüklenebilir; şekli aynı, o zaman videolar foto prompt'uyla üretilir.

Hemen altındaki hücre **üretim planını** basar: hangi kare üretilecek, hangisi neden atlanacak. Tablo, 36 GiB'lık indirmeden önce çıkar.

In [ ]:
# === Plan dosyasını yükle (video.json ya da ham export) + biçimini doğrula ===
# The uploaded file is the work order: whatever you upload is what gets rendered. Nothing is
# picked from Drive, so a stale setting can never point the run at another project.
from google.colab import files
import json, os

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError(f"❌ Tek dosya yükle — {len(uploaded)} dosya geldi: {list(uploaded)}")

PLAN_NAME = next(iter(uploaded))
try:
    PLAN_JSON = json.loads(uploaded[PLAN_NAME].decode("utf-8"))
except (UnicodeDecodeError, json.JSONDecodeError) as e:
    raise RuntimeError(f"❌ {PLAN_NAME} okunamadı: {type(e).__name__}: {e}") from None

def validate_plan(data):
    """Fail-loud on a file that is not a Queen Editor export / converted plan. Only `prompt` is
    read, so a raw export works too -- what has to exist is folder, file and prompt."""
    if not isinstance(data, dict):
        raise RuntimeError(f"❌ JSON bir nesne değil: {type(data).__name__}")
    for key in ("folder", "photos"):
        if key not in data:
            raise RuntimeError(f"❌ '{key}' alanı yok — dosyadaki anahtarlar: {sorted(data)}")
    if not isinstance(data["photos"], list):
        raise RuntimeError(f"❌ 'photos' liste değil: {type(data['photos']).__name__}")
    for i, photo in enumerate(data["photos"]):
        if not isinstance(photo, dict):
            raise RuntimeError(f"❌ photos[{i}] nesne değil: {type(photo).__name__}")
        missing = [k for k in ("file", "prompt") if k not in photo]
        if missing:
            raise RuntimeError(f"❌ photos[{i}] eksik alan: {missing} — var olanlar: {sorted(photo)}")

validate_plan(PLAN_JSON)

PHOTO_DIR = PLAN_JSON["folder"]
if not os.path.isdir(PHOTO_DIR):
    raise RuntimeError(f"❌ Fotoğraf klasörü yok: {PHOTO_DIR}\n"
                       "Drive aynı hesapla mı mount edildi? Proje silinmiş olabilir.")

# The project name comes from the file itself, so the output folder can never disagree with the
# photos it was built from.
PROJECT    = os.path.basename(PHOTO_DIR.rstrip("/"))
OUTPUT_DIR = f"{DRIVE_ROOT}/{PROJECT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ {PLAN_NAME}: {len(PLAN_JSON['photos'])} kare")
print(f"✓ Proje: {PROJECT}")
print(f"✓ Fotoğraflar (yalnız okunur): {PHOTO_DIR}")
print(f"✓ Videolar: {OUTPUT_DIR}")

In [ ]:
# === Üretim planı — indirmeden önce, bilerek ===
# loop_maker's rule: decide the whole run before a GPU minute is spent. A stale plan file or an
# already-complete output folder shows up here, not after ~40 minutes of model downloads.
# log()/human() belong to the next section and are not defined yet, so this cell prints plainly.
import os

def out_path(stem, v):
    """Output path for photo <stem>, seed-variant v (0-indexed).
    VARIANTS==1 -> no variant suffix (1_a.mp4); else 1-indexed suffix (1_a_1.mp4, 1_a_2.mp4).
    The stem comes from the photo's own file name, so 1_a.png lands as 1_a.mp4 and the video
    stays traceable back to the photo that produced it."""
    return f"{OUTPUT_DIR}/{stem}.mp4" if VARIANTS == 1 else f"{OUTPUT_DIR}/{stem}_{v + 1}.mp4"

def build_plan(photos, folder):
    """One row per (photo, seed-variant) -> (stem, v, action, image_path, prompt, reason).
    Each photo carries its own prompt: the pairing was done upstream, nothing is matched here."""
    rows = []
    for photo in photos:
        stem   = os.path.splitext(photo["file"])[0]
        image  = os.path.join(folder, photo["file"])
        prompt = photo["prompt"]
        for v in range(VARIANTS):
            out = out_path(stem, v)
            if not prompt.strip():
                rows.append((stem, v, "ATLA", None, "", "prompt boş"))
            elif not os.path.exists(image):
                # The plan file lists a photo the project no longer has -- it went stale.
                rows.append((stem, v, "ATLA", None, prompt, "fotoğraf klasörde yok"))
            elif os.path.exists(out) and os.path.getsize(out) > 0:
                rows.append((stem, v, "ATLA", image, prompt, "çıktı zaten var"))
            else:
                rows.append((stem, v, "ÜRET", image, prompt, ""))
    return rows

PLAN = build_plan(PLAN_JSON["photos"], PHOTO_DIR)

print(f"\n{'ÇIKTI':>11}  {'KARAR':<6}  {'FOTOĞRAF':<16}  AÇIKLAMA")
print("-" * 80)
for stem, v, action, image, prompt, reason in PLAN:
    disp = stem if VARIANTS == 1 else f"{stem}_{v + 1}"
    name = os.path.basename(image) if image else "—"
    detail = reason if reason else prompt.strip().replace("\n", " ")[:34]
    print(f"{disp:>11}  {action:<6}  {name:<16}  {detail}")

_to_render = sum(1 for r in PLAN if r[2] == "ÜRET")
print("-" * 80)
print(f"Üretilecek: {_to_render}  |  Atlanacak: {len(PLAN) - _to_render}")

if _to_render == 0:
    raise RuntimeError("❌ Üretilecek video yok — yukarıdaki tabloya bak (prompt boş, fotoğraf yok ya da hepsi zaten üretilmiş)")

## 3) Ortak Yardımcılar

`log` + fail-loud `run` + model doğrulama + ComfyUI hata çözümleyici.

İlk satırda **CONFIG çalıştı mı** diye bakılır: 3. ve 4. bölümün CONFIG'e ihtiyacı yok (ComfyUI yerel diske iner, Drive gerekmez), o yüzden bu kapı olmasa CONFIG'de patlayan bir hatayı ancak ~5 dakikalık kurulumdan sonra 5. bölümde görürsün.

Model doğrulaması **ağa soru sormaz**: safetensors header'ındaki `data_offsets` beklenen boyutu verir. HEAD/`Content-Length` kullanılmaz — HF'in Xet CDN'i imzalı URL'de HEAD'e 403 döner ve o hata gövdesi "dosya boyutu" sanılır.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
# Used by section 4 (custom nodes), 5 (model download) and 7 (render); defined once (DRY).
import os, json, time, struct, subprocess

# Sections 3 and 4 do not need CONFIG (ComfyUI installs to a hardcoded local path, no Drive), so
# without this gate a failed CONFIG cell stays invisible until section 5 -- after a ~5 min install.
assert "COMFY_ROOT" in globals(), "❌ Önce 1) CONFIG hücresini çalıştır — Drive mount edilmemiş"

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for download failures: the downloader exits non-zero on an HTTP error, on a
    transfer that ends before the announced length, and on a full disk.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (HF's Xet CDN answers HEAD with 403
    while serving the GET fine, so a HEAD-based size check reads the error body as the size).

    ok      -> header parses and the file is exactly as long as its header says
    partial -> valid prefix, shorter than expected: safe to resume
    invalid -> empty / error page / longer than expected: garbage, stop
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

def describe_comfy_error(status):
    """Raw execution_error from ComfyUI's history status -> (text, traceback, is_infra).

    is_infra = the failing node is a model loader, so the model is broken or missing and every
    render would hit the identical error.
    """
    for entry in status.get("messages", []):
        if not (isinstance(entry, (list, tuple)) and len(entry) == 2):
            continue
        kind, data = entry
        if kind != "execution_error" or not isinstance(data, dict):
            continue
        node_type = str(data.get("node_type", "?"))
        text = (f"node {data.get('node_id')} ({node_type})\n"
                f"{data.get('exception_type')}: {str(data.get('exception_message', '')).strip()}\n"
                f"inputs: {data.get('current_inputs')}")
        tb = "".join(data.get("traceback", []) or [])
        return text, tb, node_type.lower().endswith("loader")
    return f"status: {json.dumps(status, ensure_ascii=False)}", "", False

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors, describe_comfy_error)")

## 4) ComfyUI + Custom Node'lar (16)

Liste `wan22-arbuzai/manual.ipynb` ile birebir aynı — API grafiği aynı grafiğin IMAGE2VIDEO grubunun export'u, dolayısıyla aynı node class'larına ihtiyacı var.

Biri başarısız olursa hücre `RuntimeError` ile durur (fail-loud); eksik node ileride "node not found" olarak karşımıza çıkmaz.

In [ ]:
%cd /content

# === System deps + ComfyUI ===
!apt-get install -y ffmpeg aria2 > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python imageio imageio-ffmpeg sageattention

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides to the v5.0 graph
CUSTOM_NODES = [
    ("ComfyUI-Manager",                 "https://github.com/ltdrdata/ComfyUI-Manager.git"),            # detect missing nodes in the UI
    ("rgthree-comfy",                   "https://github.com/rgthree/rgthree-comfy.git"),               # Power Lora Loader, Seed, Fast Groups Bypasser, Label
    ("comfy_mtb",                       "https://github.com/melMass/comfy_mtb.git"),                   # Note Plus, Pick From Batch, RIFEInterpolation
    ("ComfyUI-VideoHelperSuite",        "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),# VHS_VideoCombine
    ("ComfyUI-MMAudio",                 "https://github.com/kijai/ComfyUI-MMAudio.git"),               # AUDIO2VIDEO group (its models are not downloaded)
    ("ComfyUI-WanVideoWrapper",         "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),       # Wan video nodes
    ("ComfyUI-GGUF",                    "https://github.com/city96/ComfyUI-GGUF.git"),                 # UnetLoaderGGUF (present in the graph but unset)
    ("ComfyUI-KJNodes",                 "https://github.com/kijai/ComfyUI-KJNodes.git"),               # ImageResizeKJv2, ColorMatch
    ("ComfyMath",                       "https://github.com/evanspearman/ComfyMath.git"),              # ComfyMathExpression (seconds -> frames: a*16+1)
    ("ComfyUI-Frame-Interpolation",     "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"),# RIFE
    ("ComfyUI-VFI",                     "https://github.com/GACLove/ComfyUI-VFI.git"),                 # frame interpolation
    ("ComfyUI_Comfyroll_CustomNodes",   "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git"),# CR Float To Integer
    ("ComfyUI-Easy-Use",                "https://github.com/yolain/ComfyUI-Easy-Use.git"),             # easy cleanGpuUsed
    ("ComfyUI-mxToolkit",               "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),         # mxSlider2D (resolution)
    ("ComfyUI-NAG",                     "https://github.com/scottmudge/ComfyUI-NAG.git"),              # KSamplerWithNAG (Advanced)
    ("comfyui-adaptiveprompts",         "https://github.com/Alectriciti/comfyui-adaptiveprompts.git"), # PromptGenerator
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    run(["git", "clone", "--depth", "1", url, name], f"clone {name}", timeout=120)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## 5) Modeller — önce gated probe, sonra indir (~36 GiB)

Gated erişim **ağır indirmeden önce** doğrulanır (ilk 1 KB): cookie ölmüşse 27 GiB'lık checkpoint indirmeye başlamadan, Civitai'nin **gerçek yanıtıyla** durur.

Bozuk/eksik inen dosyada hücre `RuntimeError` ile durur; bozuk dosya **silinmez**, inceleme için diskte kalır.

Dosyalar kaynağın kendi adıyla değil, **grafiğin istediği adla** iner: UNETLoader **197**/**186** `SmoothMix_I2V_v2_High/Low.safetensors`, VAELoader **191** `Wan2_1_VAE_fp32.safetensors`, Power Lora Loader **201**/**200** ise `wan2.2_i2v_lightx2v_...` ve `SmoothMix_Animations_XXX_...` arıyor. Ad tutmazsa render "model bulunamadı" ile düşer.

**Distill LoRA'lar iniyor** çünkü export'ta 201/200 dolu: I2V v2.0'da lightx2v checkpoint'e merge **edilmemiş**.

**İnmeyen (bilerek):** T2V checkpoint'leri, `clip_vision_h.safetensors`, MMAudio dosyaları — API grafiğinde hiçbiri yok.

In [ ]:
import os, glob

# === Target folders ===
COMFY = COMFY_ROOT
DIFF  = f"{COMFY}/models/diffusion_models"
LORA  = f"{COMFY}/models/loras"
for d in ["diffusion_models", "loras", "text_encoders", "vae"]:
    os.makedirs(f"{COMFY}/models/{d}", exist_ok=True)

# === Single download function — shared flow for HF (aria2c) and Civitai (curl) (DRY) ===
def fetch(url, target_dir, filename, label, *, parallel, headers=None):
    """Download + validate a model; anything invalid stops the run (fail-loud, nothing deleted).

    parallel=True -> aria2c (fast for large HF files), False -> curl (Civitai, login cookie).
    Downloads land in <target>.part and are renamed only once check_safetensors says "ok", so
    ComfyUI never sees a half-written file under the real model name.

    On failure the raw HTTP exchange is printed, not a summary of it: curl runs with
    --fail-with-body (non-zero exit, but the response body is kept instead of discarded) and -D
    (every response header of the redirect chain), so a Civitai 401/403 shows the server's own
    headers and body verbatim.
    """
    target = os.path.join(target_dir, filename)
    part = target + ".part"
    hdrs = f"/tmp/{filename}.headers"

    if os.path.exists(target):
        state, msg = check_safetensors(target)
        if state == "ok":
            log(f"{label}: zaten var ({msg})")
            return
        raise RuntimeError(f"{label}: {state} — {msg}\n{target}\n--- file head ---\n{head_text(target)}")

    resume = False
    if os.path.exists(part):
        state, msg = check_safetensors(part)
        if state == "invalid":
            # Resuming onto garbage would append good bytes to it and hide the problem.
            raise RuntimeError(f"{label}: .part {state} — {msg}\n{part}\n--- file head ---\n{head_text(part)}")
        if state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
            resume = True

    if not os.path.exists(part) or resume:
        log(f"{label}: iniyor")
        if parallel:
            cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--continue=true",
                   "--console-log-level=warn", "--auto-file-renaming=false",
                   "--allow-overwrite=true", "-d", target_dir, "-o", os.path.basename(part)]
            if headers:
                cmd += ["--header", headers]
        else:
            cmd = ["curl", "-L", "-C", "-", "--fail-with-body", "--max-time", "1800",
                   "-D", hdrs, "-o", part]
            if headers:
                cmd += ["-H", headers]
        cmd.append(url)
        try:
            run(cmd, label, timeout=3600)
        except RuntimeError as e:
            raise RuntimeError(
                f"{e}\n{url.split('?')[0]}\n"
                f"--- response headers ---\n{head_text(hdrs)}\n"
                f"--- response body ---\n{head_text(part)}"
            ) from None

    state, msg = check_safetensors(part)
    if state != "ok":
        raise RuntimeError(f"{label}: {state} — {msg}\n{part}\n{url.split('?')[0]}\n"
                           f"--- response headers ---\n{head_text(hdrs)}\n"
                           f"--- file head ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

# Civitai auth: session cookie ONLY. A ?token= API key authenticates the request as that key's
# account -> creator-gated assets answer 401.
# Host = civitai.RED: the cookie is same-origin there. Sending it to .com is cross-domain and
# returns the login+turnstile page instead of the file.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

def civitai_probe(version_id, label):
    """Fail-fast: range-download the first 1KB to verify gated access BEFORE ~27GiB of checkpoints.
    On non-2xx or a login wall, surface Civitai's ACTUAL response body -- no hardcoded guesses.
    """
    out = "/content/_probe.bin"
    code = (run(["curl", "-sL", "--max-time", "60", "-r", "0-1023",
                 "-H", cookie_header(), "-w", "%{http_code}", "-o", out,
                 civitai_url(version_id)], f"probe {label}") or "").strip()[-3:]
    body = b""
    if os.path.exists(out):
        with open(out, "rb") as f:
            body = f.read(512)
        os.remove(out)
    # success = 2xx AND the body is real binary (safetensors), not an HTML/JSON error page
    if code.startswith("2") and not body.startswith(b"<") and not body.startswith(b'{"'):
        log(f"{label}: erişim OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: HTTP {code} — Civitai yanıtı: "
                       f"{body.decode('utf-8', 'replace').strip() or '(boş gövde — binary değil)'}")

# === HuggingFace models (aria2c) ===
WAN22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
WAN21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"

HF_MODELS = [
    # (url, target_dir, filename, label)
    # The exported graph has both distill LoRAs enabled in Power Lora Loader 201/200, so they have
    # to be on disk before the render -- I2V v2.0 does not have lightx2v merged.
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "Lightx2v I2V HIGH"),
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",  LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",  "Lightx2v I2V LOW"),
    # VAE: VAELoader 191 asks for 'Wan2_1_VAE_fp32.safetensors', so land the base under that name
    (f"{WAN21}/vae/wan_2.1_vae.safetensors",                          f"{COMFY}/models/vae",           "Wan2_1_VAE_fp32.safetensors",            "Wan2.1 VAE"),
    (f"{WAN21}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", f"{COMFY}/models/text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "UMT5-XXL"),
]

# === Civitai gated models (curl + login cookie) ===
# Civitai serves these under its own file names; UNETLoader 197/186 ask for
# SmoothMix_I2V_v2_High/Low.safetensors and Power Lora Loader 201/200 for the Animations pair,
# so every file lands under the name the graph names.
CIVITAI_MODELS = [
    # (version_id, target_dir, filename, label)
    (2513182, DIFF, "SmoothMix_I2V_v2_High.safetensors",         "SmoothMix I2V v2 HIGH"),
    (2513186, DIFF, "SmoothMix_I2V_v2_Low.safetensors",          "SmoothMix I2V v2 LOW"),
    (2376136, LORA, "SmoothMix_Animations_XXX_High.safetensors", "SmoothMix Animations XXX HIGH"),
    (2376143, LORA, "SmoothMix_Animations_XXX_Low.safetensors",  "SmoothMix Animations XXX LOW"),
]

# 1) Fail-fast: verify gated access before spending half an hour on downloads
log(f"Gated probe: {len(CIVITAI_MODELS)} asset")
for vid, d, fn, label in CIVITAI_MODELS:
    civitai_probe(vid, label)

# 2) HuggingFace
for url, d, fn, label in HF_MODELS:
    fetch(url, d, fn, label, parallel=True)

# 3) Civitai — parallel=False: aria2c forwards the cookie to the B2 store on redirect and gets 403,
#    curl drops it cross-host and gets through.
for vid, d, fn, label in CIVITAI_MODELS:
    fetch(civitai_url(vid), d, fn, label, parallel=False, headers=cookie_header())

# === Summary (reaching here means everything downloaded + validated) ===
print("\n📂 diffusion_models/")
for f in sorted(glob.glob(f"{DIFF}/*.safetensors")):
    print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
print("📂 loras/")
for f in sorted(glob.glob(f"{LORA}/*.safetensors")):
    print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 6) ComfyUI'yi Başlat (Arka Planda)

ComfyUI subprocess olarak başlar, üretim localhost API'sine konuşur. **90 sn içinde hazır olmazsa** hücre log'un son 30 satırını basıp durur — sonraki hücre ölü sunucuya çalışmasın.

Tünel yok: UI'a girilmiyor. Grafiği değiştirmek istersen `wan22-arbuzai/manual.ipynb`'yi çalıştır, UI'da düzenle, **Workflow → Export (API)** ile Drive'daki `workflow_api.json`'un üzerine yaz.

Bu hücre **bloklamaz**; biter ve 7. bölüme geçilir.

In [ ]:
import subprocess, time, os, urllib.request

# Re-run safety: kill the previous instance before starting a new one
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

# === Start in background (logs to file) ===
# No --enable-manager: nothing opens the UI here, and a missing node already failed loudly during
# install. No tunnel either -- the render cell talks to localhost.
comfy_log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_ROOT, stdout=comfy_log, stderr=subprocess.STDOUT,
)
log(f"ComfyUI başlatıldı (PID {proc.pid}), log: {COMFY_LOG}")

# === Ready? max 90s — otherwise fail-loud with the server's own log ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"{COMFYUI_URL}/system_stats", timeout=2)
        log(f"ComfyUI hazır ({(i + 1) * 2}s)", "OK")
        break
    except Exception:
        pass
else:
    with open(COMFY_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")

## 7) Üret

Plan tablosunda **ÜRET** yazan her çıktı sırayla işlenir: fotoğraf Queen Editor'ün klasöründen okunup ComfyUI'ya (foto başına bir kez) yüklenir, render edilir, `queen-tools/<proje>/` altına yazılır (`N_<harf>.mp4` veya `N_<harf>_<v>.mp4`), ComfyUI'daki kopyalar silinir.

Grafiğe yazılan üç alan: LoadImage **287** (o fotoğraf), PromptGenerator **233:240** (prompt + seed), Seed **210**. LoRA'lar, ağırlıklar, çözünürlük, step, cfg — hepsi grafikten gelir.

**Yarıda kalırsa** notebook'u baştan çalıştır: çıktısı olanlar hem plan hücresinde hem döngü içinde atlanır, kaldığı yerden devam eder.

**Hata olursa:** model yükleyici hatası batch'i durdurur (her video aynı hatayı alırdı). Tek videoya özgü hata yalnız onu atlar; üst üste 3 hata batch'i durdurur. Bir video 30 dakikada bitmezse `TimeoutError` ile durulur — kalanları üretmek için notebook'u tekrar çalıştırman yeter.

Seed her varyant için ayrı üretilir ve loglanır; `SEED`'e sayı verirsen varyant `v` için `SEED + v` kullanılır.

In [ ]:
import json, os, random, time, uuid, requests

class ComfyExecutionError(RuntimeError):
    """A prompt failed inside ComfyUI. Carries the raw error, plus whether it is infra-level.
    infra=True -> a model loader node failed, so the model is broken or missing."""
    def __init__(self, text, traceback_text, infra):
        super().__init__(text)
        self.text = text
        self.traceback_text = traceback_text
        self.infra = infra

# === Node ids (from workflow_api.json) ===
# Opaque strings, not numbers: "233:240" is a subgraph-flattened id.
IMAGE_NODE  = "287"       # LoadImage, inputs.image -> the uploaded file's server-side name
PROMPT_NODE = "233:240"   # PromptGenerator, inputs.prompt -> CLIPTextEncode 176
SEED_NODE   = "210"       # Seed (rgthree) -> KSamplerAdvanced 236:206 noise_seed

MAX_CONSECUTIVE_FAILURES = 3   # a batch that keeps failing is broken, not unlucky

# === Template I/O + patchers (SRP: one function injects one field) ===
def load_workflow(path):
    with open(path, encoding="utf-8") as f:
        wf = json.load(f)
    if "nodes" in wf:
        raise RuntimeError(
            "workflow_api.json UI formatında — ComfyUI'de 'Workflow → Export (API)' ile kaydet"
        )
    for node_id in (IMAGE_NODE, PROMPT_NODE, SEED_NODE):
        if node_id not in wf:
            raise RuntimeError(f"Workflow'da {node_id} node yok — graf değişmiş, node id'leri güncelle")
    return wf

def set_image(workflow, image_name):
    workflow[IMAGE_NODE]["inputs"]["image"] = image_name

def set_prompt(workflow, prompt):
    workflow[PROMPT_NODE]["inputs"]["prompt"] = prompt

def set_seed(workflow, seed):
    """Both seeds: the sampler's noise seed and PromptGenerator's own, so a rerun with the same
    seed reproduces the video even when the prompt uses wildcard syntax.

    The graph ships Seed (rgthree) at -1. rgthree randomises in the frontend widget, which does
    not exist in API mode -- sending -1 through would pin every render to the same noise.
    """
    workflow[SEED_NODE]["inputs"]["seed"] = seed
    workflow[PROMPT_NODE]["inputs"]["seed"] = seed

def produced_files(history_entry):
    """Every file this prompt wrote, as ComfyUI-relative paths.

    The graph has two savers: VHS_VideoCombine (mp4) and SaveImage (last frame png). Only the
    video goes to Drive, but both are cleared off the Colab disk so a long batch does not fill it.
    """
    paths = []
    for node_output in history_entry.get("outputs", {}).values():
        for key in ("gifs", "videos", "images"):
            for item in node_output.get(key, []):
                if item.get("type", "output") != "output":
                    continue                       # temp previews are not on disk as outputs
                paths.append(os.path.join(item.get("subfolder", ""), item["filename"]))
    return paths

# === ComfyUI HTTP client ===
class ComfyClient:
    def __init__(self, base_url):
        self.base = base_url.rstrip("/")
        self.client_id = str(uuid.uuid4())

    def upload_image(self, local_path):
        """Push one image into ComfyUI's input/ folder. Returns the name the SERVER reports.

        ComfyUI may rename on collision, so its answer is the only name guaranteed to resolve; a
        locally guessed filename would silently point LoadImage at the wrong image.
        """
        name = os.path.basename(local_path)
        with open(local_path, "rb") as f:
            r = requests.post(f"{self.base}/upload/image",
                              files={"image": (name, f)},
                              data={"overwrite": "true"}, timeout=120)
        if r.status_code >= 400:
            raise RuntimeError(f"POST /upload/image -> HTTP {r.status_code}\n{r.text}")
        return r.json()["name"]

    def submit(self, workflow):
        r = requests.post(f"{self.base}/prompt",
                          json={"prompt": workflow, "client_id": self.client_id}, timeout=30)
        if r.status_code >= 400:
            raise RuntimeError(f"POST /prompt -> HTTP {r.status_code}\n{r.text}")
        data = r.json()
        if data.get("node_errors"):
            raise RuntimeError("POST /prompt -> node_errors\n"
                               + json.dumps(data["node_errors"], indent=2, ensure_ascii=False))
        return data["prompt_id"]

    def wait(self, prompt_id, timeout):
        start = time.time()
        while True:
            if time.time() - start > timeout:
                raise TimeoutError(f"prompt {prompt_id}: {timeout}s içinde bitmedi")
            history = requests.get(f"{self.base}/history/{prompt_id}", timeout=30).json()
            if prompt_id in history:
                entry = history[prompt_id]
                status = entry.get("status", {})
                if status.get("status_str") == "error":
                    raise ComfyExecutionError(*describe_comfy_error(status))
                return entry
            time.sleep(POLL_INTERVAL)

    def save_output_video(self, history_entry, save_path):
        """Pull the produced video over /view — independent of ComfyUI's output subfolder layout.
        The extension filter is what keeps SaveImage's png out of Drive."""
        for node_output in history_entry.get("outputs", {}).values():
            for key in ("gifs", "videos", "images"):
                for item in node_output.get(key, []):
                    if not item.get("filename", "").lower().endswith((".mp4", ".webm", ".mov")):
                        continue
                    r = requests.get(f"{self.base}/view", timeout=300, params={
                        "filename":  item["filename"],
                        "subfolder": item.get("subfolder", ""),
                        "type":      item.get("type", "output"),
                    })
                    r.raise_for_status()
                    with open(save_path, "wb") as f:
                        f.write(r.content)
                    return
        raise RuntimeError("history'de video çıktısı yok:\n"
                           + json.dumps(history_entry.get("outputs", {}), indent=2, ensure_ascii=False))

# === One render, end to end ===
def generate_one(client, save_path, image_name, prompt, seed):
    """One render: already-uploaded photo + prompt + seed -> save_path.
    save_path comes from process_all (out_path handles the VARIANTS naming)."""
    workflow = load_workflow(WORKFLOW_PATH)
    set_image(workflow, image_name)
    set_prompt(workflow, prompt)
    set_seed(workflow, seed)

    prompt_id = client.submit(workflow)
    history = client.wait(prompt_id, TIMEOUT_PER_RENDER)
    client.save_output_video(history, save_path)

    for rel in produced_files(history):
        local = os.path.join(COMFY_OUTPUT_DIR, rel)
        if os.path.exists(local):
            os.remove(local)
    return save_path

# === The batch ===
def process_all(plan):
    """Render every ÜRET row in order. Skips, failures and the reason for each are printed.

    A loader failure stops the batch: the model is broken or missing, so every remaining video
    would hit the identical error. A video-specific failure only costs that render.
    """
    todo = [row for row in plan if row[2] == "ÜRET"]
    client = ComfyClient(COMFYUI_URL)
    uploaded = {}   # image_path -> server-side name; each photo uploads once, reused by its variants
    done = skipped = failed = 0
    consecutive = 0
    t_batch = time.time()

    log(f"Batch başlıyor — {len(todo)} video")
    for stem, v, _action, image_path, prompt, _reason in todo:
        save_path = out_path(stem, v)
        disp = os.path.splitext(os.path.basename(save_path))[0]
        # Re-check the disk: an earlier run of this cell may have produced it already.
        if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
            log(f"{disp}: zaten var — atlandı")
            skipped += 1
            continue

        if image_path not in uploaded:              # upload the photo once, reused by all its variants
            uploaded[image_path] = client.upload_image(image_path)

        seed = random.randint(0, 2**31 - 1) if SEED is None else SEED + v
        log(f"{disp}: {os.path.basename(image_path)}  seed={seed}  |  "
            f"{prompt.strip()[:45]}{'…' if len(prompt.strip()) > 45 else ''}")
        t0 = time.time()
        try:
            path = generate_one(client, save_path, uploaded[image_path], prompt, seed)
        except ComfyExecutionError as e:
            print(e.text)
            print(e.traceback_text)
            if e.infra:
                raise RuntimeError(
                    f"Altyapı hatası ({e.text.splitlines()[0]}) — batch durduruldu, kalan videolar denenmedi"
                ) from None
            failed += 1
            consecutive += 1
            log(f"{disp}: başarısız — atlanıyor ({consecutive}/{MAX_CONSECUTIVE_FAILURES})", "ERR")
            if consecutive >= MAX_CONSECUTIVE_FAILURES:
                raise RuntimeError(f"Üst üste {consecutive} video başarısız — batch durduruldu") from None
            continue

        consecutive = 0
        done += 1
        log(f"{disp}: bitti ({time.time() - t0:.0f}s, {os.path.getsize(path) / 1024**2:.1f} MB) → {path}", "OK")

    log(f"Batch bitti ({(time.time() - t_batch) / 60:.0f} dk) — "
        f"üretildi: {done}, atlandı: {skipped}, başarısız: {failed}", "OK")

process_all(PLAN)